In [ ]:
!pip install openai

In [ ]:

import os
import re
import json
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from items import Item
from evaluator import evaluate
from google.colab import userdata


In [ ]:

# # Fine-tune OpenAI GPT Model for Price Prediction

# Load environment variables
load_dotenv(override=True)
hf_token = userdata.get('HF_TOKEN')
bs_router = userdata.get('BSROUTEER')
api_key = userdata.get('API_KEY')
login(hf_token, add_to_git_credential=True)

# -----------------------------
# 2️⃣ Load dataset
# -----------------------------
LITE_MODE = False
username = "ed-donner"
dataset_name = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"


In [ ]:
print(api_key, bs_router)

In [ ]:

train, val, test = Item.from_hub(dataset_name)
print(f"Loaded {len(train):,} train, {len(val):,} val, {len(test):,} test items")
print(api_key, bs_router)
# -----------------------------
# 3️⃣ Initialize OpenAI client
# -----------------------------
openai = OpenAI(
    base_url=bs_router,
    api_key=api_key
)

# -----------------------------
# 4️⃣ Prepare data for fine-tuning
# -----------------------------
def messages_for(item):
    """
    Prepare chat messages for each item:
    - User asks for price estimation
    - Assistant provides price formatted as $xx.xx
    """
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]

def make_jsonl(items):
    """
    Convert list of items to JSONL string for OpenAI fine-tuning
    """
    return "\n".join(json.dumps({"messages": messages_for(item)}) for item in items)

def write_jsonl(items, filename):
    """
    Write JSONL file to disk
    """
    with open(filename, "w") as f:
        f.write(make_jsonl(items))


In [ ]:

# Subset for fine-tuning (cost-efficient)
fine_tune_train = train[:100]
fine_tune_validation = val[:50]

# Create JSONL files
os.makedirs("jsonl", exist_ok=True)
write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")
write_jsonl(fine_tune_validation, "jsonl/fine_tune_validation.jsonl")

# -----------------------------
# 5️⃣ Upload files to OpenAI
# -----------------------------
with open("jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

print("Training file ID:", train_file.id)
print("Validation file ID:", validation_file.id)

# -----------------------------
# 6️⃣ Create fine-tuning job
# -----------------------------
ft_job = openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="pricer"
)

job_id = ft_job.id
print("Fine-tuning job ID:", job_id)

# -----------------------------
# 7️⃣ Monitor fine-tuning
# -----------------------------
# Retrieve job details
job_info = openai.fine_tuning.jobs.retrieve(job_id)
print("Fine-tuned model name:", job_info.fine_tuned_model)

# View recent job events
events = openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data
for e in events:
    print(e.timestamp, e.message)

# -----------------------------
# 8️⃣ Inference function
# -----------------------------
fine_tuned_model_name = job_info.fine_tuned_model



In [ ]:
def test_messages_for(item):
    """
    Prepare prompt for inference
    """
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

def gpt_fine_tuned_predict(item):
    """
    Generate price prediction using the fine-tuned GPT model
    """
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=7
    )
    # Extract raw text
    return response.choices[0].message.content.strip()

# -----------------------------
# 9️⃣ Test predictions
# -----------------------------
print("Ground price:", test[0].price)
print("Predicted price:", gpt_fine_tuned_predict(test[0]))

# -----------------------------
# 10️⃣ Evaluate model on test set
# -----------------------------
evaluate(gpt_fine_tuned_predict, test)